# Build a Search Engine Using YouTube Video Data

**Group members:** _Mohammad Irsheid  and   Basil Farah_

This notebook builds a simple Information Retrieval system over 400 YouTube videos about Artificial Intelligence. It processes the text, builds an inverted index, answers Boolean queries, uses query optimization and skip pointers, and supports exact two-word phrases with a biword index. The core algorithms are implemented from scratch.

## Imports

Only modules from Python's standard library are used. The notebook must stay in the `notebooks` folder, because the dataset is loaded from `../data/Dataset.json`.

In [ ]:
import json
import math
import re
import time
from pathlib import Path

DATASET_PATH = Path("../data/Dataset.json")

## Activity 1: The document collection

The dataset contains 400 YouTube videos on the theme of Artificial Intelligence, stored as JSON. Each video has six fields: `video_id`, `title`, `description`, `channel`, `published_date` and `url`. The `video_id` is unique, so it is used as the document ID in the whole project.

In [ ]:
with open(DATASET_PATH, "r", encoding="utf-8") as file:
    data = json.load(file)

print("Number of videos:", len(data))
print(data[0])

Number of videos: 400
{'video_id': 'ZfC98aM9C8Y', 'title': "The 'Godfather of AI' predicts humanity's chance of extinction", 'description': 'Listen to the full show on the all-new LBC App: https://app.af.lbc.co.uk/btnc/thenewlbcapp #benkentish #artificialintelligence\xa0...', 'channel': 'LBC', 'published_date': '2026-11-09', 'url': 'https://www.youtube.com/watch?v=ZfC98aM9C8Y'}


## Activity 2: Text processing

Each title and description is converted to lowercase, punctuation is replaced with spaces, and the text is split into words. Repeated words are handled with `unique_words`, which keeps every word of the title and description once. The `video_id` stays with each document.

The ordered word lists are kept as well, because the biword index in Activity 7 needs to know which words are next to each other. The channel is processed in the same way, but it is not part of `unique_words`, so it is not searchable.

In [ ]:
TEXT_FIELDS = ["title", "description", "channel"]

lowercased_data = []
for item in data:
    new_item = item.copy()
    for field in TEXT_FIELDS:
        new_item[field] = re.sub(r"[^\w\s]", " ", new_item[field].lower())
        new_item[field] = new_item[field].split()

    combined_words = new_item["title"] + new_item["description"]
    new_item["unique_words"] = list(dict.fromkeys(combined_words))

    lowercased_data.append(new_item)

if __name__ == "__main__":
    first = lowercased_data[:5]
    for item in first:
        print(item["title"], "||")
        print("Unique words:", item["unique_words"], "\n")

['the', 'godfather', 'of', 'ai', 'predicts', 'humanity', 's', 'chance', 'of', 'extinction'] ||
Unique words: ['the', 'godfather', 'of', 'ai', 'predicts', 'humanity', 's', 'chance', 'extinction', 'listen', 'to', 'full', 'show', 'on', 'all', 'new', 'lbc', 'app', 'https', 'af', 'co', 'uk', 'btnc', 'thenewlbcapp', 'benkentish', 'artificialintelligence'] 

['artificial', 'intelligence', 'full', 'course', 'artificial', 'intelligence', 'tutorial', 'for', 'beginners', 'edureka'] ||
Unique words: ['artificial', 'intelligence', 'full', 'course', 'tutorial', 'for', 'beginners', 'edureka', 'pgp', 'in', 'generative', 'ai', 'and', 'ml', 'collaboration', 'with', 'illinois', 'tech'] 

['what', 'is', 'ai', 'artificial', 'intelligence', 'what', 'is', 'artificial', 'intelligence', 'ai', 'in', '5', 'mins', 'simplilearn'] ||
Unique words: ['what', 'is', 'ai', 'artificial', 'intelligence', 'in', '5', 'mins', 'simplilearn', 'microsoft', 'engineer', 'program'] 

['ai', 'simplified', '6', 'concepts', 'you', 'n

## Activity 3: Inverted index

The inverted index maps every term to the videos that contain it. For each term it stores:

- `df`: the document frequency, which is the number of videos containing the term
- `doc_ids`: the posting list, sorted and without duplicates
- `skips`: the skip pointers used in Activity 6, created while the index is built

A set removes duplicate IDs, and sorting makes it possible to merge two lists efficiently. `show_inverted_index` displays the contents.

In [ ]:
def create_skips(doc_ids, min_length=10):
    skips = {}
    if len(doc_ids) >= min_length:
        step = int(math.sqrt(len(doc_ids)))
        for i in range(0, len(doc_ids) - step, step):
            skips[i] = i + step
    return skips

def build_inverted_index(documents):
    temp_index={}
    for doc in documents:
        doc_id= doc["video_id"]
        for term in doc["unique_words"]:
            if term not in temp_index:
                temp_index[term] = set()
            temp_index[term].add(doc_id)

    inverted_index = {}
    for term in sorted(temp_index.keys()):
        sorted_doc_ids = sorted(temp_index[term])
        inverted_index[term] = {
            "df": len(sorted_doc_ids),
            "doc_ids": sorted_doc_ids,
            "skips": create_skips(sorted_doc_ids)
        }
    return inverted_index

def show_inverted_index(inverted_index, terms=None):
    print(f"Number of unique terms in the inverted index:{len(inverted_index)}")
    if terms:
        terms_to_display = [t.lower() for t in terms if t.lower() in inverted_index]
    else:
        terms_to_display = sorted(
            inverted_index.keys(),
            key=lambda x: inverted_index[x]["df"],
            reverse=True
        ) [:10]
    for term in terms_to_display:
        print(f"Term: {term}")
        print(f"Document Frequency (df): {inverted_index[term]['df']}")
        print(f"Document IDs: {inverted_index[term]['doc_ids']}\n")

if __name__ == "__main__":
    inverted_index = build_inverted_index(lowercased_data)
    show_inverted_index(inverted_index, terms=["artificial", "intelligence", "machine","learning", "data", "science"])

Number of unique terms in the inverted index:2108
Term: artificial
Document Frequency (df): 59
Document IDs: ['-9bo8HlSxwQ', '-sB12gk9ESA', '0jspaMLxBig', '1X-rr1DKSbY', '3xZSU7hVHUY', '4RixMPF4xis', '5NgNicANyqM', '8fqHZnVUiWk', '94p9QNtSn0A', '9gpvus-fPyo', '9iZvFzaNwwM', '9vM4p9NN0Ts', 'CBhqnitNorI', 'De2t1tDQ8AE', 'DzpHeXVSC5I', 'E-HSXRvL9Ik', 'EUMMxVRbI6A', 'GGY17VD_9Bs', 'JMLsHI8aV0g', 'JMUxmLyrhSk', 'JcXKbUIebrU', 'KKNCiRWd_j0', 'KlG0VyhIzBU', 'LGCZ-Fhm48c', 'MfIjxPh6Pys', 'SSE4M0gcmvE', 'TjZBTDzGeGg', 'Ts5Z64A0Vv4', 'U0L2yVXhpJY', 'U5zQzSqywEY', 'WsdcF7EEvhM', 'XMxcSf9XkbQ', 'XXaUd0fGpOs', 'Yq0QkCxoTHM', '_19pRsZRiz4', '_NLHFoVNlbg', 'aZ5EsdnpLMI', 'ad79nYk2keg', 'b2q5OFtxm6A', 'c0m6yaGlZh4', 'gMicQCf3UXE', 'jGwO_UgTS7I', 'm8o2GrbR3d8', 'm94OMx1eBy0', 'mUw27wG7uFA', 'ml57_bSSDF4', 'oV3ZY6tJiA0', 'ojOdB-7Dbdg', 'qD-o2lDQwa8', 'qYNweeDHiyU', 'qrvK_KuIeJk', 'r_Q_-gf7bR4', 'rbCQKODKv1o', 'sg_fuEzFw0g', 'tFx_UNW9I1U', 'ttIOdAdQaUE', 'ws_9ettrtSg', 'xjjkBczuYwM', 'z-EtmaFJieY']

Term

## Activity 4: Query parser

The parser works in three steps:

1. **Tokenize:** a regular expression separates words, operators and parentheses. Any other symbol raises an error.
2. **Validate:** the tokens are read from left to right while tracking whether an operand or an operator is expected. This detects queries such as `artificial AND` or `artificial NOT`.
3. **Convert to postfix:** the Shunting-Yard algorithm uses a stack to apply the precedence: parentheses, then `NOT`, then `AND`, then `OR`. It also detects mismatched parentheses.

The test below contains 5 valid and 9 invalid queries.

In [ ]:
OPERATORS = ["AND", "OR", "NOT"]
PRECEDENCE = {"NOT": 3, "AND": 2, "OR": 1}
def tokenize_query(query):
    raw_tokens= re.findall(r'\w+|AND|OR|NOT|\(|\)|\S', query)
    tokens = []
    for token in raw_tokens:
        upper_token= token.upper()
        if upper_token in OPERATORS or token in ("(", ")"):
            tokens.append(upper_token)
        else:
            cleaned_token= re.sub(r"[^\w\s]", "", token.lower())
            if cleaned_token:
                tokens.append(cleaned_token)
            else:
                raise ValueError(f"Invalid token: '{token}'.")
    return tokens

def validate_tokens(tokens):
    expect_operand = True
    for token in tokens:
        if token=="(" or token == "NOT":
            if not expect_operand:
                raise ValueError(f"Expected AND or OR before {token}.")
        elif token==")":
            if expect_operand:
                raise ValueError(f"Expected a word before ')'.")
        elif token in OPERATORS:
            if expect_operand:
                raise ValueError(f"Expected a word before '{token}'.")
            expect_operand = True
        else:
            if not expect_operand:
                raise ValueError(f"Expected AND or OR before '{token}'.")
            expect_operand = False
    if expect_operand:
        raise ValueError("The query is empty or incomplete.")

def parse_to_postfix(query):

    tokens =  tokenize_query(query)
    validate_tokens(tokens)
    output_queue = []
    operator_stack = []

    for token in tokens:
        if token=="(":
            operator_stack.append(token)
        elif token==")":
            while operator_stack and operator_stack[-1]!="(":
                output_queue.append(operator_stack.pop())
            if not operator_stack:
                raise ValueError("Mismatched parentheses in the query.")
            operator_stack.pop()  
        elif token in OPERATORS:
            while( token!= "NOT" and operator_stack and operator_stack[-1]!="(" and PRECEDENCE[operator_stack[-1]] >= PRECEDENCE[token]):
                output_queue.append(operator_stack.pop())
            operator_stack.append(token)
        else:
            output_queue.append(token)

    while operator_stack:
        top = operator_stack.pop()
        if top in ("(", ")"):
            raise ValueError("Mismatched parentheses in the query.")
        output_queue.append(top)
    return output_queue

if __name__ == "__main__":
    query = ["artificial AND intelligence",
             "machine OR deep",
             "artificial AND NOT robotics",
             "(machine OR deep) AND learning",
             "NOT NOT data", 
             "artificial AND",
             "AND artificial",
             "artificial NOT",
             "artificial machine AND",
             "artificial intelligence",
             "()",
             "",
             "(artificial AND machine",
             "artificial AND & machine"]
    for q in query:
        print(f"Query: {q}")
        try:
            postfix_expression = parse_to_postfix(q)
            print("Valid query. Postfix:", postfix_expression)
        except ValueError as e:
            print(f"Invalid query: {e}")

Query: artificial AND intelligence
Valid query. Postfix: ['artificial', 'intelligence', 'AND']
Query: machine OR deep
Valid query. Postfix: ['machine', 'deep', 'OR']
Query: artificial AND NOT robotics
Valid query. Postfix: ['artificial', 'robotics', 'NOT', 'AND']
Query: (machine OR deep) AND learning
Valid query. Postfix: ['machine', 'deep', 'OR', 'learning', 'AND']
Query: NOT NOT data
Valid query. Postfix: ['data', 'NOT', 'NOT']
Query: artificial AND
Invalid query: The query is empty or incomplete.
Query: AND artificial
Invalid query: Expected a word before 'AND'.
Query: artificial NOT
Invalid query: Expected AND or OR before NOT.
Query: artificial machine AND
Invalid query: Expected AND or OR before 'machine'.
Query: artificial intelligence
Invalid query: Expected AND or OR before 'intelligence'.
Query: ()
Invalid query: Expected a word before ')'.
Query: 
Invalid query: The query is empty or incomplete.
Query: (artificial AND machine
Invalid query: Mismatched parentheses in the quer

## Activity 5: Boolean search

The postfix query is evaluated with a stack. A word pushes its posting list, and an operator takes lists from the stack and pushes the result. A word that is not in the index gives an empty list.

`AND`, `OR` and `NOT` walk through two sorted lists with two pointers. If the IDs are different, the pointer at the smaller ID moves forward. For each query, the output shows the original query, the number of matches, and the ID, title and URL of every matching video.

In [ ]:
def intersect_postings(postings1, postings2):
    result = []
    i, j = 0, 0
    while i < len(postings1) and j < len(postings2):
        if postings1[i] == postings2[j]:
            result.append(postings1[i])
            i += 1
            j += 1
        elif postings1[i] < postings2[j]:
            i += 1
        else:
            j += 1
    return result

def union_postings(postings1, postings2):
    result = []
    i, j = 0, 0
    while i < len(postings1) and j < len(postings2):
        if postings1[i] == postings2[j]:
            result.append(postings1[i])
            i += 1
            j += 1
        elif postings1[i] < postings2[j]:
            result.append(postings1[i])
            i += 1
        else:
            result.append(postings2[j])
            j += 1
    result.extend(postings1[i:])
    result.extend(postings2[j:])
    return result

def not_postings(postings, all_doc_ids):
    result = []
    i,j = 0,0
    while i < len(all_doc_ids) and j < len(postings):
        if all_doc_ids[i] == postings[j]:
            i += 1
            j += 1
        elif all_doc_ids[i] < postings[j]:
            result.append(all_doc_ids[i])
            i += 1
        else:
            j += 1
    result.extend(all_doc_ids[i:])
    return result

def boolean_search(query, inverted_index, all_doc_ids):
    postfix_expression = parse_to_postfix(query)
    stack = []

    for token in postfix_expression:
        match token:
            case "AND":
                if len(stack) < 2:
                    raise ValueError("Invalid syntax: 'AND' requires two operands.")
                right = stack.pop()
                left = stack.pop()
                result = intersect_postings(left, right)
                stack.append(result)
            case "OR":
                if len(stack) < 2:
                    raise ValueError("Invalid syntax: 'OR' requires two operands.")
                right = stack.pop()
                left = stack.pop()
                result = union_postings(left, right)  
                stack.append(result)
            case "NOT":
                if len(stack) < 1:
                    raise ValueError("Invalid syntax: 'NOT' requires one operand.")
                operand = stack.pop()
                result = not_postings(operand, all_doc_ids)
                stack.append(result)
            case _:
                if token in inverted_index:
                    postings = inverted_index[token]["doc_ids"]
                    stack.append(postings)
                else:
                    stack.append([])

    if len(stack) != 1:
        raise ValueError("Invalid syntax, it's missing an operator between terms.")
    return stack[0]

def show_search_results(query, matching_doc_ids, video_lookup):
    print(f"Query: {query}")
    print(f"Number of matching documents: {len(matching_doc_ids)}")

    for rank, vid_id in enumerate(matching_doc_ids, start=1):
        meta= video_lookup.get(vid_id, {})
        title = meta.get("title", "Unknown Title")
        url = meta.get("url", f"https://www.youtube.com/watch?v={vid_id}")
        print(f"{rank}. [{vid_id}] {title}")
        print(f"   URL: {url}")
    print("-" * 40)

if __name__ == "__main__":
    dataset_path = DATASET_PATH
    with open(dataset_path, "r", encoding="utf-8") as file:
        raw_videos = json.load(file)

    video_lookup = {video["video_id"]: video for video in raw_videos}
    all_doc_ids = sorted(video_lookup.keys())
    index = build_inverted_index(lowercased_data)

    test_queries = [
        "artificial AND intelligence",
        "machine OR robotics",
        "artificial AND NOT robotics",
        "(machine OR deep) AND learning"
    ]

    for query in test_queries:
        matches = boolean_search(query, index, all_doc_ids)
        show_search_results(query, matches, video_lookup)

Query: artificial AND intelligence
Number of matching documents: 58
1. [-9bo8HlSxwQ] CS50x - Artificial Intelligence
   URL: https://www.youtube.com/watch?v=-9bo8HlSxwQ
2. [-sB12gk9ESA] A.I. Revolution | Full Documentary | NOVA | PBS
   URL: https://www.youtube.com/watch?v=-sB12gk9ESA
3. [0jspaMLxBig] Andrew Ng: Deep Learning, Education, and Real-World AI | Lex Fridman Podcast #73
   URL: https://www.youtube.com/watch?v=0jspaMLxBig
4. [1X-rr1DKSbY] Elon Musk on AI: humans will no longer be in control in ten years  | The Economist
   URL: https://www.youtube.com/watch?v=1X-rr1DKSbY
5. [3xZSU7hVHUY] HOW TO LEARN & Master AI in 2026 ? (Complete Powerful 7-step ROADMAP)
   URL: https://www.youtube.com/watch?v=3xZSU7hVHUY
6. [4RixMPF4xis] AI vs Machine Learning
   URL: https://www.youtube.com/watch?v=4RixMPF4xis
7. [5NgNicANyqM] Harvard CS50’s Artificial Intelligence with Python – Full University Course
   URL: https://www.youtube.com/watch?v=5NgNicANyqM
8. [8fqHZnVUiWk] Computer Vision Cla

## Activity 6: Query optimization and skip pointers

**Part A, query optimization:** the terms of an `AND` query are sorted by document frequency, so the shortest posting list is processed first. The search stops early if the result becomes empty.

**Part B, skip pointers:** posting lists with at least 10 videos get a skip pointer about every √L positions, where L is the length of the list. During `AND`, if the ID at the skip target is not larger than the ID in the other list, the search jumps there. Intermediate results have no skip pointers.

**Counting:** each turn of the loop counts as one comparison, and each check of a skip target counts as one more. Both versions use the same rule. Each query runs in four ways, and the results are checked against the Boolean search.

In [ ]:
def intersect_count(postings1, postings2):
    result = []
    comparisons = 0
    i, j = 0, 0
    while i < len(postings1) and j < len(postings2):
        if postings1[i] == postings2[j]:
            result.append(postings1[i])
            i += 1
            j += 1
        elif postings1[i] < postings2[j]:
            i += 1
        else:
            j += 1
        comparisons += 1
    return result, comparisons

def intersect_with_skips(postings1, skips1, postings2, skips2):
    result = []
    comparisons = 0
    i, j = 0, 0
    while i < len(postings1) and j < len(postings2):
        if postings1[i] == postings2[j]:
            result.append(postings1[i])
            i += 1
            j += 1
        elif postings1[i] < postings2[j]:
            if i in skips1:
                comparisons += 1
                if postings1[skips1[i]] <= postings2[j]:
                    i = skips1[i]
                else:
                    i += 1
            else:
                i += 1
        else:
            if j in skips2:
                comparisons += 1
                if postings2[skips2[j]] <= postings1[i]:
                    j = skips2[j]
                else:
                    j += 1
            else:
                j += 1
        comparisons += 1
    return result, comparisons

def and_search(query, index, optimize, use_skips):
    tokens = tokenize_query(query)
    validate_tokens(tokens)

    for token in tokens:
        if token in ("OR", "NOT", "(", ")"):
            raise ValueError("and_search only supports AND.")

    words = []
    for token in tokens:
        if token != "AND":
            words.append(token)

    # Piece 4: shortest list first
    empty = {"df": 0, "doc_ids": [], "skips": {}}
    if optimize:
        words.sort(key=lambda word: index.get(word, empty)["df"])

    # Piece 5: start with the first word
    first = index.get(words[0], empty)
    result = first["doc_ids"]
    result_skips = first["skips"]
    total = 0

    # Piece 6: intersect with each remaining word
    for word in words[1:]:
        if not result:
            break
        entry = index.get(word, empty)
        if use_skips:
            result, comparisons = intersect_with_skips(result, result_skips, entry["doc_ids"], entry["skips"])
        else:
            result, comparisons = intersect_count(result, entry["doc_ids"])
        result_skips = {}
        total += comparisons

    return words, result, total


if __name__ == "__main__":
    
    index = build_inverted_index(lowercased_data)
    all_doc_ids = sorted(doc["video_id"] for doc in lowercased_data)

    test_queries = [
        "artificial AND intelligence",
        "artificial AND intelligence AND robotics",
        "machine AND learning",
        "ai AND robotics",
        "ai AND learning AND neural"
    ]

    for query in test_queries:
        _, result1, normal = and_search(query, index, False, False)
        _, result2, skips = and_search(query, index, False, True)
        order, result3, optimized = and_search(query, index, True, False)
        _, result4, both = and_search(query, index, True, True)
        expected = boolean_search(query, index, all_doc_ids)
        same = result1 == result2 == result3 == result4 == expected
        print(f"Query: {query}")
        print(f"Optimized order: {order}")
        print(f"Matching documents: {len(expected)}")
        print(f"Comparisons: normal={normal}, skips={skips}, optimized={optimized}, both={both}")
        print(f"Same results as Boolean search: {same}")
        print("-" * 40)

Query: artificial AND intelligence
Optimized order: ['artificial', 'intelligence']
Matching documents: 58
Comparisons: normal=60, skips=60, optimized=60, both=60
Same results as Boolean search: True
----------------------------------------
Query: artificial AND intelligence AND robotics
Optimized order: ['robotics', 'artificial', 'intelligence']
Matching documents: 0
Comparisons: normal=87, skips=87, optimized=27, both=13
Same results as Boolean search: True
----------------------------------------
Query: machine AND learning
Optimized order: ['machine', 'learning']
Matching documents: 69
Comparisons: normal=121, skips=126, optimized=121, both=126
Same results as Boolean search: True
----------------------------------------
Query: ai AND robotics
Optimized order: ['robotics', 'ai']
Matching documents: 1
Comparisons: normal=104, skips=21, optimized=104, both=21
Same results as Boolean search: True
----------------------------------------
Query: ai AND learning AND neural
Optimized order

**Result:** skip pointers do not always reduce the number of comparisons. They help when one list is very short and the other is long: `ai AND robotics` drops from 104 to 21. When many videos match, the skip checks can cost more than they save: `machine AND learning` rises from 121 to 126. Optimization helps when there are three or more terms: `artificial AND intelligence AND robotics` drops from 87 to 27, and to 13 together with skip pointers.

## Activity 7: Biword index

A biword is a pair of neighbouring words. "Introduction to Machine Learning" gives `introduction to`, `to machine` and `machine learning`. Each biword is stored as an index term with its document frequency and sorted video IDs.

The title and the description are processed separately, so the last word of the title is never paired with the first word of the description. `phrase_search` accepts exactly two words in double quotes. The test compares searching for both words using `AND` with searching for the exact phrase.

In [ ]:
def make_biwords(words):
    biwords = []
    for i in range(len(words) - 1):
        biwords.append(words[i] + " " + words[i + 1])
    return biwords

def build_biword_index(documents):
    temp_index = {}
    for doc in documents:
        doc_id = doc["video_id"]
        for field in ["title", "description"]:
            for biword in make_biwords(doc[field]):
                if biword not in temp_index:
                    temp_index[biword] = set()
                temp_index[biword].add(doc_id)

    biword_index = {}
    for biword in sorted(temp_index.keys()):
        sorted_doc_ids = sorted(temp_index[biword])
        biword_index[biword] = {
            "df": len(sorted_doc_ids),
            "doc_ids": sorted_doc_ids
        }
    return biword_index

def phrase_search(query, biword_index):
    query = query.strip()
    if len(query) < 2 or not (query.startswith('"') and query.endswith('"')):
        raise ValueError("A phrase must be inside double quotes.")

    words = re.findall(r"\w+", query.lower())
    if len(words) != 2:
        raise ValueError("A phrase must have exactly two words.")

    biword = words[0] + " " + words[1]
    if biword in biword_index:
        return biword_index[biword]["doc_ids"]
    return []

if __name__ == "__main__":
    index = build_inverted_index(lowercased_data)
    biword_index = build_biword_index(lowercased_data)
    all_doc_ids = sorted(doc["video_id"] for doc in lowercased_data)
    titles = {doc["video_id"]: " ".join(doc["title"]) for doc in lowercased_data}

    print(make_biwords(["introduction", "to", "machine", "learning"]))
    print(f"Number of unique biwords: {len(biword_index)}")
    print("-" * 40)

    test_phrases = [
        ("machine", "learning"),
        ("artificial", "intelligence"),
        ("data", "science"),
        ("computer", "science"),
        ("ai", "tutorial"),
        ("ai", "course")
    ]

    for word1, word2 in test_phrases:
        and_result = boolean_search(f"{word1} AND {word2}", index, all_doc_ids)
        phrase_result = phrase_search(f'"{word1} {word2}"', biword_index)
        only_and = [doc_id for doc_id in and_result if doc_id not in phrase_result]

        print(f"{word1} AND {word2}: {len(and_result)} videos")
        print(f'"{word1} {word2}": {len(phrase_result)} videos')
        print(f"Found only by AND: {len(only_and)}")
        for doc_id in only_and[:3]:
            print(f"  [{doc_id}] {titles[doc_id]}")
        print("-" * 40)

['introduction to', 'to machine', 'machine learning']
Number of unique biwords: 6097
----------------------------------------
machine AND learning: 69 videos
"machine learning": 69 videos
Found only by AND: 0
----------------------------------------
artificial AND intelligence: 58 videos
"artificial intelligence": 58 videos
Found only by AND: 0
----------------------------------------
data AND science: 16 videos
"data science": 15 videos
Found only by AND: 1
  [z-EtmaFJieY] machine learning artificial intelligence crash course computer science 34
----------------------------------------
computer AND science: 12 videos
"computer science": 7 videos
Found only by AND: 5
  [67uVfZmUQP8] computer vision explained for beginners
  [U0L2yVXhpJY] what is computer vision artificial intelligence for beginners tamil karthik s show
  [jVwvnWRYcdk] computer vision explained
----------------------------------------
ai AND tutorial: 37 videos
"ai tutorial": 13 videos
Found only by AND: 24
  [-qiIihY5v

**Result:** every phrase match is also an `AND` match, but not the other way around. For `machine learning` and `artificial intelligence` there is no difference in this dataset. For more loosely related words the difference is large: `ai AND course` finds 46 videos, but only 10 contain the phrase "ai course".

## Activity 8: Evaluation

The evaluation uses 12 Boolean queries and 5 phrases. Part 1 compares standard and optimized search. For `AND` queries, optimized means ordering by document frequency plus skip pointers. For queries with `OR`, `NOT` or parentheses, it means skip pointers only. Part 2 compares `AND` search with biword phrase search.

In the tables, `cmp` is the number of comparisons and `us` is the search time in microseconds, averaged over 1000 runs with `time.perf_counter()`. `Same` shows whether both searches return the same videos as the Boolean search from Activity 5.

In [ ]:
def union_count(postings1, postings2):
    result = []
    comparisons = 0
    i, j = 0, 0
    while i < len(postings1) and j < len(postings2):
        if postings1[i] == postings2[j]:
            result.append(postings1[i])
            i += 1
            j += 1
        elif postings1[i] < postings2[j]:
            result.append(postings1[i])
            i += 1
        else:
            result.append(postings2[j])
            j += 1
        comparisons += 1
    result.extend(postings1[i:])
    result.extend(postings2[j:])
    return result, comparisons

def not_count(postings, all_doc_ids):
    result = []
    comparisons = 0
    i, j = 0, 0
    while i < len(all_doc_ids) and j < len(postings):
        if all_doc_ids[i] == postings[j]:
            i += 1
            j += 1
        elif all_doc_ids[i] < postings[j]:
            result.append(all_doc_ids[i])
            i += 1
        else:
            j += 1
        comparisons += 1
    result.extend(all_doc_ids[i:])
    return result, comparisons

def mixed_search(query, index, all_doc_ids, use_skips):
    empty = {"df": 0, "doc_ids": [], "skips": {}}
    stack = []
    total = 0
    for token in parse_to_postfix(query):
        if token == "NOT":
            postings, skips = stack.pop()
            result, comparisons = not_count(postings, all_doc_ids)
            stack.append((result, {}))
            total += comparisons
        elif token == "OR":
            right, right_skips = stack.pop()
            left, left_skips = stack.pop()
            result, comparisons = union_count(left, right)
            stack.append((result, {}))
            total += comparisons
        elif token == "AND":
            right, right_skips = stack.pop()
            left, left_skips = stack.pop()
            if use_skips:
                result, comparisons = intersect_with_skips(left, left_skips, right, right_skips)
            else:
                result, comparisons = intersect_count(left, right)
            stack.append((result, {}))
            total += comparisons
        else:
            entry = index.get(token, empty)
            stack.append((entry["doc_ids"], entry["skips"]))
    return stack[0][0], total

def measure_time(function, *inputs):
    runs = 1000
    start = time.perf_counter()
    for _ in range(runs):
        function(*inputs)
    end = time.perf_counter()
    return (end - start) / runs * 1_000_000

if __name__ == "__main__":
    index = build_inverted_index(lowercased_data)
    biword_index = build_biword_index(lowercased_data)
    all_doc_ids = sorted(doc["video_id"] for doc in lowercased_data)

    and_queries = [
        "artificial AND intelligence",
        "machine AND learning",
        "ai AND robotics",
        "artificial AND intelligence AND robotics",
        "ai AND learning AND neural",
        "the AND to AND networks"
    ]

    mixed_queries = [
        "machine OR robotics",
        "NOT data",
        "artificial AND NOT robotics",
        "(machine OR deep) AND learning",
        "(neural OR robotics) AND ai",
        "ai AND robotics OR machine AND neural"
    ]

    phrases = [
        ("machine", "learning"),
        ("data", "science"),
        ("computer", "science"),
        ("ai", "tutorial"),
        ("ai", "course")
    ]

    print("PART 1: standard and optimized search")
    print(f"{'Query':<42}{'Matches':>8}{'Std cmp':>9}{'Opt cmp':>9}{'Std us':>9}{'Opt us':>9}  Same")
    for query in and_queries + mixed_queries:
        if query in and_queries:
            _, standard_result, standard = and_search(query, index, False, False)
            _, optimized_result, optimized = and_search(query, index, True, True)
            standard_time = measure_time(and_search, query, index, False, False)
            optimized_time = measure_time(and_search, query, index, True, True)
        else:
            standard_result, standard = mixed_search(query, index, all_doc_ids, False)
            optimized_result, optimized = mixed_search(query, index, all_doc_ids, True)
            standard_time = measure_time(mixed_search, query, index, all_doc_ids, False)
            optimized_time = measure_time(mixed_search, query, index, all_doc_ids, True)

        expected = boolean_search(query, index, all_doc_ids)
        same = standard_result == optimized_result == expected
        print(f"{query:<42}{len(expected):>8}{standard:>9}{optimized:>9}{standard_time:>9.1f}{optimized_time:>9.1f}  {same}")

    print()
    print("PART 2: AND search and biword phrase search")
    print(f"{'Words':<26}{'AND':>6}{'Phrase':>8}{'Only AND':>10}{'AND cmp':>9}{'AND us':>9}{'Phrase us':>11}")
    for word1, word2 in phrases:
        and_query = f"{word1} AND {word2}"
        phrase = f'"{word1} {word2}"'
        _, and_result, comparisons = and_search(and_query, index, False, False)
        phrase_result = phrase_search(phrase, biword_index)
        and_time = measure_time(and_search, and_query, index, False, False)
        phrase_time = measure_time(phrase_search, phrase, biword_index)
        only_and = len(and_result) - len(phrase_result)
        print(f"{word1 + ' ' + word2:<26}{len(and_result):>6}{len(phrase_result):>8}{only_and:>10}{comparisons:>9}{and_time:>9.1f}{phrase_time:>11.1f}")

PART 1: standard and optimized search
Query                                      Matches  Std cmp  Opt cmp   Std us   Opt us  Same
artificial AND intelligence                     58       60       60     10.9     12.0  True
machine AND learning                            69      121      126     18.3     20.3  True


ai AND robotics                                  1      104       21     16.0      7.1  True


artificial AND intelligence AND robotics         0       87       13     16.3      7.4  True
ai AND learning AND neural                       5      333      239     45.9     35.9  True


the AND to AND networks                          3      376      189     51.1     29.3  True
machine OR robotics                             71       36       36      9.8      9.6  True


NOT data                                       374      399      399     60.6     62.9  True


artificial AND NOT robotics                     59      582      582     85.9     92.1  True


(machine OR deep) AND learning                 114      244      246     42.7     43.7  True
(neural OR robotics) AND ai                      6      177      163     29.5     28.4  True


ai AND robotics OR machine AND neural            8      173       99     30.2     20.5  True

PART 2: AND search and biword phrase search
Words                        AND  Phrase  Only AND  AND cmp   AND us  Phrase us
machine learning              69      69         0      121     17.9        1.1
data science                  16      15         1       34      7.5        1.1
computer science              12       7         5       71     12.3        1.1
ai tutorial                   37      13        24      226     31.5        1.1
ai course                     46      10        36      254     33.3        1.1


## Report

**Standard and optimized search.** Every optimized search returned exactly the same videos as the Boolean search. Of the 12 queries, 6 needed fewer comparisons, 4 needed the same number and 2 needed more. The improvement is largest when a query contains a rare term: `artificial AND intelligence AND robotics` dropped from 87 to 13 comparisons, because `robotics` appears in only one video. `the AND to AND networks` dropped from 376 to 189.

**When optimization does not help.** With two terms, changing the order does not change the number of comparisons. When many videos match, checking the skip pointers can cost more than it saves: `machine AND learning` rose from 121 to 126 and `(machine OR deep) AND learning` from 244 to 246. `OR` and `NOT` must go through their lists, so they do not use skip pointers: `artificial AND NOT robotics` needed 582 comparisons both times.

**Search time.** Fewer comparisons usually meant less time. `artificial AND intelligence AND robotics` went from 16.3 to 7.4 microseconds, and `the AND to AND networks` from 51.1 to 29.3. Time and comparisons are not exactly proportional, because sorting the terms and looking up skip pointers also take time. `artificial AND intelligence` needed 60 comparisons in both versions, but took 10.9 and 12.0 microseconds. The times vary slightly between runs.

**AND search and biword search.** The phrase search is more precise. `ai AND course` found 46 videos, but only 10 contain the phrase "ai course". For `machine learning` both searches found the same 69 videos. The phrase search is also faster, at about 1.1 microseconds compared with 7.5 to 33.3 for `AND`, because it looks up the pair directly and does not merge posting lists. The cost is a larger index: 6097 biwords compared with 2108 single terms.

**Limitations**

- The results are not ranked by relevance.
- Only exact words match, so `network` does not find `networks`.
- Most descriptions in the dataset are short snippets that are cut off.
- Phrases must have exactly two words and cannot be combined with Boolean operators.
- The words `and`, `or` and `not` cannot be searched, because they are read as operators.
- The dataset is small, so the time differences are only a few microseconds.